In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
def resolve_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, path.parent):
        if (candidate / "data" / "processed_data").exists():
            return candidate
    raise FileNotFoundError("Could not find data/processed_data.")


processed_dir = resolve_project_root() / "data" / "processed_data"

life_df = pd.read_csv(processed_dir / "processed_life_expectancy_total.csv")
life_df["year"] = pd.to_numeric(life_df["year"], errors="coerce")
life_df["value"] = pd.to_numeric(life_df["value"], errors="coerce")
life_df = life_df.dropna(
    subset=["year", "value", "country_code", "income_group"]
).copy()

1. Time series: average life expectancy by income group

In [4]:
output_dir = Path("images")
output_dir.mkdir(exist_ok=True)

income_group_trends = (
    life_df.groupby(["income_group", "year"], as_index=False)["value"].mean()
)

fig_time = px.line(
    income_group_trends,
    x="year",
    y="value",
    color="income_group",
    title="Life expectancy at birth by income group (1960–2023)",
    labels={
        "year": "Year",
        "value": "Life expectancy at birth (years)",
        "income_group": "Income group",
    },
)
fig_time.update_layout(legend_title_text="Income group", template="plotly_white")
fig_time.write_image(output_dir / "life_expectancy_by_income_group.png", width=1200, height=700)
fig_time.show()

2. World map: life expectancy in 2023

In [5]:
output_dir = Path("images")
output_dir.mkdir(exist_ok=True)

life_2023 = life_df[life_df["year"] == 2023].dropna(subset=["country_code"]).copy()

fig_map = px.choropleth(
    life_2023,
    locations="country_code",
    locationmode="ISO-3",
    color="value",
    hover_name="country_name",
    title="Life expectancy at birth by country in 2023",
    color_continuous_scale="Viridis",
    range_color=[life_2023["value"].min(), life_2023["value"].max()],
    labels={"value": "Life expectancy (years)"},
)
fig_map.update_layout(margin={"r": 10, "t": 40, "l": 10, "b": 10})
fig_map.write_image(output_dir / "life_expectancy_world_map_2023.png", width=1400, height=700)
fig_map.show()

3. Sankey: movement between life-expectancy buckets (1960 → 2023)

In [6]:
output_dir = Path("images")
output_dir.mkdir(exist_ok=True)

bucket_labels = [
    "Very low life expectancy",
    "Low life expectancy",
    "Medium life expectancy",
    "High life expectancy",
    "Very high life expectancy",
]


def assign_bucket(series: pd.Series) -> pd.Series:
    return pd.qcut(
        series,
        q=5,
        labels=bucket_labels,
        duplicates="drop",
    )


life_1960 = life_df[life_df["year"] == 1960].dropna(subset=["country_code", "value"]).copy()
life_2023_buckets = life_df[life_df["year"] == 2023].dropna(subset=["country_code", "value"]).copy()

life_1960["bucket_1960"] = assign_bucket(life_1960["value"])
life_2023_buckets["bucket_2023"] = assign_bucket(life_2023_buckets["value"])

transition_df = life_1960[["country_name", "country_code", "bucket_1960"]].merge(
    life_2023_buckets[["country_name", "country_code", "bucket_2023"]],
    on=["country_name", "country_code"],
    how="inner",
)

flow = (
    transition_df.groupby(["bucket_1960", "bucket_2023"], as_index=False, observed=False)
    .size()
    .rename(columns={"size": "count"})
)

node_labels = bucket_labels + [f"{label} (2023)" for label in bucket_labels]
node_index = {label: i for i, label in enumerate(bucket_labels)}
node_index_2023 = {label: i + len(bucket_labels) for i, label in enumerate(bucket_labels)}

fig_sankey = go.Figure(
    data=[
        go.Sankey(
            node=dict(label=node_labels, pad=15, thickness=20),
            link=dict(
                source=[node_index[row["bucket_1960"]] for _, row in flow.iterrows()],
                target=[node_index_2023[row["bucket_2023"]] for _, row in flow.iterrows()],
                value=flow["count"].tolist(),
            ),
        )
    ]
)
fig_sankey.update_layout(
    title_text="Country transitions between life expectancy buckets (1960 → 2023)",
    font_size=11,
)
fig_sankey.write_image(output_dir / "life_expectancy_sankey_1960_2023.png", width=1400, height=800)
fig_sankey.show()